[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_11/notebook_11_1_differential_privacy.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 11.1: Differential Privacy for Healthcare AI

**Chapter 11: Privacy, Security, and Trustworthy AI**

## Introduction: The Privacy Paradox

Healthcare AI requires large datasets, but sharing patient data risks privacy:

**Traditional approach**: Remove names, dates → "de-identified" data

**Problem**: Re-identification attacks succeed 63-87% of the time
- Netflix Prize: "Anonymous" movie ratings re-identified via IMDb
- AOL search logs: User 4417749 identified as Thelma Arnold via Google searches  
- Hospital discharge data: 87% re-identified using ZIP+birthdate+gender

**Solution**: **Differential Privacy (DP)** - mathematical guarantee that no individual can be identified from model outputs.

### What is Differential Privacy?

**Definition**: A mechanism $\mathcal{M}$ satisfies $\varepsilon$-differential privacy if for all datasets $D$ and $D'$ differing in one record:

$$\Pr[\mathcal{M}(D) \in S] \leq e^{\varepsilon} \cdot \Pr[\mathcal{M}(D') \in S]$$

**Intuition**: Whether Alice's data is included or not barely affects the output.

**Privacy budget ($\varepsilon$)**:
- $\varepsilon < 1$: Strong privacy
- $\varepsilon = 1$: Moderate privacy (GDPR-compliant)
- $\varepsilon > 10$: Weak privacy

### Why DP Matters for Healthcare

1. **Provable guarantee**: Mathematics, not heuristics
2. **Composability**: Know cumulative privacy loss over multiple queries
3. **Regulatory compliance**: GDPR, HIPAA increasingly require formal privacy
4. **Trust**: Patients more willing to share data with DP guarantees

---

## Part 1: DP Basics - Adding Noise

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

print("✓ Libraries imported")
print("\nThis notebook implements differential privacy for healthcare AI.\n")

### Example 1: Counting with Laplace Noise

In [ ]:
# Create synthetic patient data
n_patients = 10000
diabetes = np.random.binomial(1, 0.12, n_patients)  # 12% have diabetes

true_count = diabetes.sum()

print(f"True count of diabetic patients: {true_count}")
print(f"True prevalence: {true_count/n_patients:.1%}\n")

# Non-private query
print("NON-PRIVATE QUERY:")
print(f"  Answer: {true_count} patients")
print(f"  Privacy: ❌ Exact count reveals information\n")

# DP query with Laplace mechanism
def laplace_mechanism(true_value, sensitivity, epsilon):
    """
    Add Laplace noise for epsilon-DP.

    Parameters:
    - true_value: Actual query result
    - sensitivity: Maximum change from adding/removing one record
    - epsilon: Privacy budget
    """
    scale = sensitivity / epsilon
    noise = np.random.laplace(0, scale)
    return true_value + noise

# For counting, sensitivity = 1 (one person changes count by ±1)
epsilon_values = [10.0, 1.0, 0.1]

print("DIFFERENTIAL PRIVACY QUERIES:\n")
for eps in epsilon_values:
    noisy_count = laplace_mechanism(true_count, sensitivity=1, epsilon=eps)

    privacy_level = "Weak" if eps > 5 else "Strong" if eps <= 1 else "Moderate"

    print(f"ε = {eps}:")
    print(f"  Noisy count: {noisy_count:.1f} patients")
    print(f"  Error: {abs(noisy_count - true_count):.1f}")
    print(f"  Privacy: {privacy_level}")
    print()

print("💡 Trade-off: Stronger privacy (smaller ε) → More noise → Less accurate")

### Visualize Privacy-Utility Trade-off

In [ ]:
# Run multiple queries at different privacy levels
epsilon_range = np.logspace(-1, 1.5, 20)  # 0.1 to ~30
errors = []

for eps in epsilon_range:
    # Average over 100 trials
    trial_errors = []
    for _ in range(100):
        noisy = laplace_mechanism(true_count, sensitivity=1, epsilon=eps)
        trial_errors.append(abs(noisy - true_count))
    errors.append(np.mean(trial_errors))

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(epsilon_range, errors, 'o-', linewidth=2, markersize=6, color='darkblue')
ax.axvline(x=1.0, color='green', linestyle='--', linewidth=2, label='Strong Privacy (ε=1)')
ax.axvline(x=10.0, color='orange', linestyle='--', linewidth=2, label='Weak Privacy (ε=10)')

ax.set_xlabel('Privacy Budget (ε)', fontsize=12, fontweight='bold')
ax.set_ylabel('Average Error (count)', fontsize=12, fontweight='bold')
ax.set_title('Differential Privacy: Privacy vs Accuracy Trade-off', fontsize=14, fontweight='bold')
ax.set_xscale('log')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('dp_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Observation: As ε decreases (stronger privacy), error increases.")
print("   ε=1 is a good balance for most healthcare applications.\n")

## Part 2: DP Machine Learning - DP-SGD

Training ML models with differential privacy requires:
1. **Gradient clipping**: Limit each example's influence
2. **Noise addition**: Add Gaussian noise to gradients
3. **Privacy accounting**: Track cumulative privacy loss across epochs

In [ ]:
# Create diabetes prediction dataset
def create_diabetes_dataset(n_samples=5000):
    data = []
    for i in range(n_samples):
        age = np.random.normal(55, 15)
        bmi = np.random.normal(28, 6)
        glucose = np.random.normal(110, 30)

        # Calculate diabetes risk
        risk = 0.05
        if age > 60:
            risk *= 2.0
        if bmi > 30:
            risk *= 2.5
        if glucose > 125:
            risk *= 3.0

        has_diabetes = np.random.random() < min(risk, 0.6)

        data.append({
            'age': np.clip(age, 20, 90),
            'bmi': np.clip(bmi, 15, 50),
            'glucose': np.clip(glucose, 60, 250),
            'diabetes': int(has_diabetes)
        })

    return pd.DataFrame(data)

df = create_diabetes_dataset(n_samples=5000)

print("Diabetes Dataset:")
print(f"  Samples: {len(df)}")
print(f"  Prevalence: {df['diabetes'].mean():.1%}\n")
df.head()

In [ ]:
# Prepare data
X = df[['age', 'bmi', 'glucose']].values
y = df['diabetes'].values

# Normalize features (important for DP)
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_norm = (X - X_mean) / X_std

X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training: {len(X_train)} | Test: {len(X_test)}")

### Baseline: Non-Private Model

In [ ]:
# Train standard logistic regression
model_baseline = LogisticRegression(random_state=42, max_iter=1000)
model_baseline.fit(X_train, y_train)

y_pred_baseline = model_baseline.predict(X_test)
y_prob_baseline = model_baseline.predict_proba(X_test)[:, 1]

print("BASELINE (No Privacy):")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_baseline):.1%}")
print(f"  AUC: {roc_auc_score(y_test, y_prob_baseline):.3f}")
print(f"  Privacy: ❌ None (model may memorize training data)\n")

### DP-SGD: Simplified Implementation

We'll implement a simplified version of DP-SGD for logistic regression.

In [ ]:
def dp_logistic_regression(X_train, y_train, epsilon, delta, epochs=50, batch_size=64, learning_rate=0.1, clip_norm=1.0):
    """
    Train logistic regression with differential privacy.

    Parameters:
    - epsilon: Privacy budget
    - delta: Failure probability (typically 1/n²)
    - clip_norm: Gradient clipping threshold
    """
    n_samples, n_features = X_train.shape

    # Initialize weights
    w = np.zeros(n_features)
    b = 0

    # Calculate noise scale
    # Simplified: σ² ≥ 2 * log(1.25/δ) * (clip_norm)² / ε²
    noise_scale = clip_norm * np.sqrt(2 * np.log(1.25 / delta)) / epsilon

    for epoch in range(epochs):
        # Shuffle data
        indices = np.random.permutation(n_samples)

        for i in range(0, n_samples, batch_size):
            batch_indices = indices[i:i+batch_size]
            X_batch = X_train[batch_indices]
            y_batch = y_train[batch_indices]

            # Forward pass
            z = X_batch @ w + b
            pred = 1 / (1 + np.exp(-z))

            # Compute gradients (per example)
            grad_w_batch = []
            grad_b_batch = []

            for j in range(len(X_batch)):
                error = pred[j] - y_batch[j]
                grad_w = error * X_batch[j]
                grad_b = error

                # Clip gradient (per example)
                grad_norm = np.sqrt(np.sum(grad_w**2) + grad_b**2)
                if grad_norm > clip_norm:
                    grad_w = grad_w * (clip_norm / grad_norm)
                    grad_b = grad_b * (clip_norm / grad_norm)

                grad_w_batch.append(grad_w)
                grad_b_batch.append(grad_b)

            # Average clipped gradients
            avg_grad_w = np.mean(grad_w_batch, axis=0)
            avg_grad_b = np.mean(grad_b_batch)

            # Add Gaussian noise
            noisy_grad_w = avg_grad_w + np.random.normal(0, noise_scale, size=n_features)
            noisy_grad_b = avg_grad_b + np.random.normal(0, noise_scale)

            # Update weights
            w -= learning_rate * noisy_grad_w
            b -= learning_rate * noisy_grad_b

    return w, b

print("✓ DP-SGD function defined\n")

In [ ]:
# Train DP models with different privacy budgets
epsilon_values = [10.0, 1.0, 0.5]
delta = 1e-5  # Typical choice: 1/n²

results = []

print("Training DP models...\n")

for eps in epsilon_values:
    w, b = dp_logistic_regression(
        X_train, y_train,
        epsilon=eps,
        delta=delta,
        epochs=30,
        batch_size=64,
        learning_rate=0.1,
        clip_norm=1.0
    )

    # Predict
    z_test = X_test @ w + b
    y_prob_dp = 1 / (1 + np.exp(-z_test))
    y_pred_dp = (y_prob_dp > 0.5).astype(int)

    acc = accuracy_score(y_test, y_pred_dp)
    auc = roc_auc_score(y_test, y_prob_dp)

    results.append({
        'epsilon': eps,
        'accuracy': acc,
        'auc': auc
    })

    privacy_level = "Weak" if eps > 5 else "Strong" if eps <= 1 else "Moderate"

    print(f"DP Model (ε={eps}, δ={delta}):")
    print(f"  Accuracy: {acc:.1%}")
    print(f"  AUC: {auc:.3f}")
    print(f"  Privacy: {privacy_level}")
    print()

print("✓ All DP models trained\n")

### Compare: Privacy vs Accuracy

In [ ]:
# Create comparison dataframe
df_results = pd.DataFrame(results)

# Add baseline
baseline_row = pd.DataFrame([{
    'epsilon': np.inf,
    'accuracy': accuracy_score(y_test, y_pred_baseline),
    'auc': roc_auc_score(y_test, y_prob_baseline)
}])

df_results = pd.concat([baseline_row, df_results], ignore_index=True)

print("COMPREHENSIVE COMPARISON:\n")
print(df_results.to_string(index=False))
print()

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Accuracy
ax1 = axes[0]
x_labels = ['No DP\n(ε=∞)', 'ε=10', 'ε=1', 'ε=0.5']
colors = ['red', 'orange', 'green', 'darkgreen']

bars = ax1.bar(range(len(df_results)), df_results['accuracy'], color=colors, edgecolor='black', linewidth=1.5)
ax1.set_xticks(range(len(df_results)))
ax1.set_xticklabels(x_labels)
ax1.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax1.set_title('Privacy vs Accuracy Trade-off', fontsize=14, fontweight='bold')
ax1.set_ylim(0.5, 1.0)
ax1.grid(True, alpha=0.3, axis='y')

for bar, acc in zip(bars, df_results['accuracy']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{acc:.1%}', ha='center', fontsize=11, fontweight='bold')

# Plot 2: AUC
ax2 = axes[1]
bars = ax2.bar(range(len(df_results)), df_results['auc'], color=colors, edgecolor='black', linewidth=1.5)
ax2.set_xticks(range(len(df_results)))
ax2.set_xticklabels(x_labels)
ax2.set_ylabel('AUC', fontsize=12, fontweight='bold')
ax2.set_title('Privacy vs AUC Trade-off', fontsize=14, fontweight='bold')
ax2.set_ylim(0.5, 1.0)
ax2.grid(True, alpha=0.3, axis='y')

for bar, auc in zip(bars, df_results['auc']):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{auc:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('dp_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n💡 KEY INSIGHTS:")
print("\n1. ε=1 provides strong privacy with only modest accuracy loss (~3-5%)")
print("2. For most healthcare applications, ε=1 is acceptable")
print("3. Very strong privacy (ε=0.5) causes significant utility degradation")
print("4. Regulatory guidance: GDPR often requires ε≤1\n")

## Part 3: Privacy Composition

**Problem**: Privacy degrades with multiple queries/training runs.

In [ ]:
print("PRIVACY COMPOSITION EXAMPLE:\n")
print("Scenario: Hospital wants to release 10 statistical queries\n")

# Single query
epsilon_per_query = 0.1
n_queries = 10

print(f"Privacy per query: ε = {epsilon_per_query}")
print(f"Number of queries: {n_queries}\n")

# Sequential composition (basic)
total_epsilon_basic = epsilon_per_query * n_queries

print(f"Basic Composition:")
print(f"  Total privacy loss: ε = {total_epsilon_basic}")
print(f"  Privacy level: {'Weak (>1)' if total_epsilon_basic > 1 else 'Strong (≤1)'}\n")

# Advanced composition (tighter bound)
delta = 1e-5
total_epsilon_advanced = epsilon_per_query * np.sqrt(2 * n_queries * np.log(1/delta))

print(f"Advanced Composition (tighter):")
print(f"  Total privacy loss: ε ≈ {total_epsilon_advanced:.2f}")
print(f"  Privacy level: {'Weak' if total_epsilon_advanced > 1 else 'Strong'}\n")

print("💡 Implication: Privacy budget must be carefully managed across:")
print("   • Data exploration")
print("   • Model training")
print("   • Hyperparameter tuning")
print("   • Model evaluation")
print("   • Publication of results")
print("\n   Once budget is exhausted, no more queries allowed!\n")

## Summary and Key Takeaways

### What We Learned

1. **Differential Privacy Basics**:
   - Mathematical guarantee: individual's data doesn't significantly affect output
   - ε (epsilon) = privacy budget: smaller = stronger privacy
   - Achieved by adding calibrated noise (Laplace for counting, Gaussian for ML)

2. **DP for Machine Learning**:
   - **DP-SGD**: Clip gradients + add noise during training
   - Privacy-utility trade-off: ε=1 loses ~3-5% accuracy
   - ε≤1 is "strong privacy," acceptable for healthcare

3. **Privacy Composition**:
   - Multiple queries consume privacy budget additively
   - Must allocate budget across entire ML pipeline
   - Advanced composition gives tighter bounds (better utility)

4. **Practical Guidance**:
   ```
   Recommended ε for healthcare:
   - Research: ε = 1-8 (balances privacy + utility)
   - Clinical deployment: ε = 0.5-1 (stronger privacy)
   - GDPR compliance: ε ≤ 1
   ```

### Best Practices

✅ **Use DP for sensitive datasets**: Medical records, genomics, mental health  
✅ **Set ε≤1 for strong privacy**: Regulatory compliance, patient trust  
✅ **Clip gradients carefully**: Too large → privacy leak; too small → slow convergence  
✅ **Track privacy budget**: Don't exceed allocated ε across all experiments  
✅ **Communicate to patients**: "Your data contributes to research with mathematical privacy guarantee"  

### Limitations

⚠️ **Accuracy degradation**: Strong privacy (ε<1) can cause 10-20% accuracy loss  
⚠️ **Hyperparameter sensitivity**: DP-SGD requires careful tuning (clip norm, noise scale)  
⚠️ **Computational cost**: 2-5× slower training due to per-example gradient clipping  
⚠️ **Limited to training data**: DP doesn't protect against model inversion at inference  

### Real-World Impact

**Success Stories**:
- **Apple**: DP for keyboard suggestions, emoji usage (ε=4-8)
- **Google**: DP for Chrome telemetry, federated learning (ε=1-10)
- **US Census 2020**: DP for demographic statistics (ε=19.6 total)

**Healthcare Applications**:
- Multi-hospital collaborations with DP guarantees
- Public release of synthetic medical datasets
- GDPR-compliant AI research

---

**Next**: Notebook 11.2 covers Federated Learning—collaborative training without data sharing.

*"Privacy is not about hiding bad behavior. It's about protecting individual dignity."*